# Importar codigos

Se obtienen las funciones del codigo

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import subprocess
import math
import random



In [2]:
import import_ipynb
from Recodificacion import *
from Codigos import *

In [3]:
METODOS      = ["pillow", "ffmpeg", "imagemagick"]
FORMATOS     = ["jpg", "webp", "png"]
CARPETA_RAIZ = "imagenes"
CALIDADES_POOL = [10, 30, 50, 70, 90]
CALIDADES_POR_IMAGEN = 3
SEED = 42  # reproducible — mismas calidades siempre para la misma imagen


# Funcion para obtener todos los valores cuantitativos de la imagen

Obtiene los valores seleccionados de la parte teorica

In [5]:
import os

def recodificar_imagen(
    img_path,
    metodo,
    formato,
    calidad
):

    # -----------------------------------------
    # VALIDACIONES
    # -----------------------------------------

    metodo = metodo.lower()
    formato = formato.lower()

    metodos_validos = [
        "pillow",
        "ffmpeg",
        "imagemagick"
    ]

    formatos_validos = [
        "jpg",
        "png",
        "webp"
    ]

    if metodo not in metodos_validos:
        raise ValueError(
            f"Método no soportado: {metodo}"
        )

    if formato not in formatos_validos:
        raise ValueError(
            f"Formato no soportado: {formato}"
        )

    # -----------------------------------------
    # RUTA SALIDA
    # -----------------------------------------

    nombre = os.path.splitext(
        os.path.basename(img_path)
    )[0]

    out_dir = os.path.join(
        "imagenes",
        metodo,
        formato
    )

    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(
        out_dir,
        f"{nombre}_{metodo}_q{calidad}.{formato}"
    )

    # -----------------------------------------
    # LLAMAR FUNCIÓN CORRESPONDIENTE
    # -----------------------------------------

    if metodo == "pillow":

        convert_image_pillow(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    elif metodo == "ffmpeg":

        convert_with_ffmpeg(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    elif metodo == "imagemagick":

        convert_with_imagemagick(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    # -----------------------------------------
    # VALIDAR GENERACIÓN
    # -----------------------------------------

    if not os.path.exists(out_path):

        raise RuntimeError(
            f"No se generó el archivo:\n{out_path}"
        )

    # -----------------------------------------
    # RETORNO
    # -----------------------------------------

    return out_path

In [6]:
def features_histograma(path, prefix):
    import numpy as np

    f = {}

    hist, hist_norm = obtener_histograma_intensidades(path)

    # guardar todo
    f[f"{prefix}_hist"] = hist.tolist()
    f[f"{prefix}_hist_norm"] = hist_norm.tolist()
    f[f"{prefix}_hist_bins"] = list(range(256))

    # derivados
    hist_norm_safe = hist_norm[hist_norm > 0]

    f[f"{prefix}_hist_entropy"] = -np.sum(hist_norm_safe * np.log2(hist_norm_safe))
    f[f"{prefix}_hist_std"] = np.std(hist_norm)
    f[f"{prefix}_hist_mean"] = np.mean(hist_norm)
    f[f"{prefix}_hist_min"] = np.min(hist_norm)
    f[f"{prefix}_hist_max"] = np.max(hist_norm)

    mu = np.mean(hist_norm)
    f[f"{prefix}_hist_skew"] = np.mean((hist_norm - mu)**3)
    f[f"{prefix}_hist_kurt"] = np.mean((hist_norm - mu)**4)

    return f

In [7]:
def features_generales(path, prefix):
    f = {}

    # básicas
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    # contraste
    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)

    # frecuencia
    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    # bordes multi escala
    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    # gradiente
    f[f"{prefix}_grad_mean"] = obtener_magnitud_media_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)

    # textura
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_textura(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_textura(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    # color
    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)

    return f

In [8]:
def extraer_features(path, prefix):
    f = {}

    f.update(features_generales(path, prefix))
    f.update(features_histograma(path, prefix))

    return f

In [9]:
def procesar_imagen(
    img_path,
    df,
    metodo="pillow",
    formato="jpg",
    calidad=50
):
    import pandas as pd

    if not isinstance(df, pd.DataFrame):
        raise TypeError("df debe ser un pandas.DataFrame")

    # 1. recodificar
    out_path = recodificar_imagen(img_path, metodo, formato, calidad)

    # 2. features
    feats_orig = extraer_features(img_path, "orig")
    feats_comp = extraer_features(out_path, "comp")

    # 3. métrica objetivo
    metricas = calcular_ms_ssim_lpips(img_path, out_path)

    # 4. fila
    fila = {
        "img_original":     img_path,
        "img_recodificada": out_path,
        "metodo":           metodo,
        "formato":          formato,
        "calidad":          calidad,
        **feats_orig,
        **feats_comp,
        **metricas
    }

    return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)

In [10]:
def procesar_dataset_completo(
    df: pd.DataFrame,
    formato_origen: str = "jpg",
    inicio: int = 0,
    fin: int = None
) -> pd.DataFrame:

    random.seed(SEED)

    formato_orig = formato_origen.lower().strip()

    if formato_orig not in FORMATOS:
        raise ValueError(
            f"formato_origen debe ser uno de {FORMATOS}, recibido: '{formato_orig}'"
        )

    carpeta = os.path.join(CARPETA_RAIZ, formato_orig)

    if not os.path.isdir(carpeta):
        raise FileNotFoundError(f"Carpeta no encontrada: {carpeta}")

    imagenes = sorted([
        f for f in os.listdir(carpeta)
        if f.lower().endswith(f".{formato_orig}")
    ])

    if not imagenes:
        print(f"⚠ Sin imágenes en: {carpeta}")
        return df

    formatos_salida = [f for f in FORMATOS if f != formato_orig]
    fmt_A = formatos_salida[0]
    fmt_B = formatos_salida[1]

    mitad = math.ceil(len(imagenes) / 2)
    mitad_A = imagenes[:mitad]
    mitad_B = imagenes[mitad:]

    todas = list(zip(
        mitad_A + mitad_B,
        [fmt_A] * len(mitad_A) + [fmt_B] * len(mitad_B)
    ))

    fin_real = fin if fin is not None else len(todas)
    subset = todas[inicio:fin_real]

    print(f"\n{'='*60}")
    print(f"📁 {carpeta}  ({len(imagenes)} imágenes total)")
    print(f"   Procesando [{inicio}:{fin_real}] → {len(subset)} imágenes")
    print(f"{'='*60}")

    for idx_global, (nombre_archivo, formato_sal) in enumerate(
        subset,
        start=inicio
    ):

        img_path = os.path.join(carpeta, nombre_archivo)

        metodo = METODOS[idx_global % len(METODOS)]

        # Una sola calidad aleatoria por imagen
        calidad = random.choice(CALIDADES_POOL)

        print(
            f"\n🖼 [{idx_global}] {nombre_archivo}"
            f" → {formato_sal}"
            f" [{metodo}]"
            f" calidad={calidad}"
        )

        try:
            df = procesar_imagen(
                img_path=img_path,
                df=df,
                metodo=metodo,
                formato=formato_sal,
                calidad=calidad
            )

        except Exception as e:
            print(f"❌ q={calidad} {e}")
            continue

    print(
        f"\n✅ [{inicio}:{fin_real}] completado "
        f"— filas acumuladas: {len(df)}"
    )

    return df

# Metodo objetivo

In [11]:

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

LPIPS_MIN_SIZE = 161  # tamaño mínimo recomendado para MS-SSIM

def safe_rgb(img: np.ndarray) -> np.ndarray:
    """Garantiza salida HxWx3 uint8 o float32 según entrada."""
    if img.ndim == 2:
        img = color.gray2rgb(img)
    elif img.shape[-1] == 4:
        img = color.rgba2rgb(img)      # rgba2rgb ya devuelve [0,1] float
    return img

def load_np(path: str) -> np.ndarray:
    """
    Carga imagen y la devuelve en float32 con rango [0, 255].
    Detecta imágenes ya normalizadas a [0,1] y las reescala.
    """
    img = safe_rgb(io.imread(path))

    # rgba2rgb de skimage devuelve float en [0,1]; resto viene en [0,255] uint
    if img.dtype in (np.float32, np.float64) and img.max() <= 1.0:
        img = (img * 255.0)

    return img.astype(np.float32)

def match_size_np(
    img1: np.ndarray,
    img2: np.ndarray,
    silent: bool = False
) -> tuple[np.ndarray, np.ndarray]:
    """
    Redimensiona img2 al tamaño de img1 si difieren.
    Emite advertencia por defecto — nunca falla silenciosamente.
    """
    if img1.shape != img2.shape:
        if not silent:
            warnings.warn(
                f"Tamaños distintos: {img1.shape} vs {img2.shape}. "
                "Se redimensiona la imagen comprimida — esto puede afectar la métrica.",
                UserWarning,
                stacklevel=2,
            )
        img2 = (resize(img2, img1.shape, anti_aliasing=True) * 255.0).astype(np.float32)
    return img1, img2

def _validate_size(arr: np.ndarray, name: str) -> None:
    h, w = arr.shape[:2]
    if h < LPIPS_MIN_SIZE or w < LPIPS_MIN_SIZE:
        raise ValueError(
            f"{name} demasiado pequeña ({h}×{w}). "
            f"LPIPS/VGG requiere mínimo {LPIPS_MIN_SIZE}×{LPIPS_MIN_SIZE} px."
        )


def calcular_ms_ssim_lpips(
    img_orig_path: str,
    img_comp_path: str,
    alpha: float = 0.5,
    beta: float = 0.5,
    silent_resize: bool = False,
) -> dict:

    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("alpha y beta deben estar en [0, 1].")

    arr_o = load_np(img_orig_path)
    arr_c = load_np(img_comp_path)
    arr_o, arr_c = match_size_np(arr_o, arr_c, silent=silent_resize)

    _validate_size(arr_o, "Imagen original")
    _validate_size(arr_c, "Imagen comprimida")

    with torch.no_grad():

        # ---- MS-SSIM — entrada en [0, 1]
        t_o = torch.from_numpy(arr_o / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)
        t_c = torch.from_numpy(arr_c / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)

        ms = ms_ssim(t_o, t_c, data_range=1.0, size_average=True).item()

        # ---- LPIPS — entrada en [-1, 1]
        lp_o = (t_o * 2.0) - 1.0
        lp_c = (t_c * 2.0) - 1.0

        lp_raw = loss_fn(lp_o, lp_c).item()

    # ---- Score combinado con valores reales ----
    # MS-SSIM: mayor = mejor  → se usa directo
    # LPIPS:   menor = mejor  → se invierte con 1/(1+lp)  → acotado en (0,1]
    lp_norm = 1.0 / (1.0 + lp_raw)  # siempre en (0,1], sin cap arbitrario

    score = alpha * ms + beta * lp_norm

    return {
        'ms_ssim':    round(ms, 6),
        'lpips_raw':  round(lp_raw, 6),   # valor real sin modificar
        'lpips_norm': round(lp_norm, 6),  # invertido matemáticamente, sin cap
        'combinado':  round(score, 6),    # siempre en (0,1]
    }

# Ejecutar

In [12]:
import warnings
import torch
import numpy as np
from skimage import io, color
from skimage.transform import resize
import lpips
from pytorch_msssim import ms_ssim

# ---------------------------
# CONFIGURACIÓN GLOBAL
# ---------------------------


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Carga única del modelo — costoso, no repetir por llamada
loss_fn = lpips.LPIPS(net='alex').to(device)
loss_fn.eval()
for p in loss_fn.parameters():
    p.requires_grad_(False)   # evita acumulación de grafo innecesaria


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\lpips\weights\v0.1\alex.pth


In [13]:
# Celda 1 — se ejecuta, usa RAM, termina y libera
df = pd.DataFrame()
df = procesar_dataset_completo(df, formato_origen="jpg")
df.to_csv("csv/dataset_jpg.csv", index=False)
del df  # fuerza la liberación inmediata
print("✅ JPG listo")


📁 imagenes\jpg  (8000 imágenes total)
   Procesando [0:8000] → 8000 imágenes

🖼 [0] COCO_test2014_000000000057.jpg → webp [pillow] calidad=10
Convertido: imagenes\jpg\COCO_test2014_000000000057.jpg → imagenes\pillow\webp\COCO_test2014_000000000057_pillow_q10.webp

🖼 [1] COCO_test2014_000000000080.jpg → webp [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\webp\COCO_test2014_000000000080_ffmpeg_q10.webp

🖼 [2] COCO_test2014_000000000090.jpg → webp [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\webp\COCO_test2014_000000000090_imagemagick_q50.webp

🖼 [3] COCO_test2014_000000000155.jpg → webp [pillow] calidad=30
Convertido: imagenes\jpg\COCO_test2014_000000000155.jpg → imagenes\pillow\webp\COCO_test2014_000000000155_pillow_q30.webp

🖼 [4] COCO_test2014_000000000171.jpg → webp [ffmpeg] calidad=30
[FFmpeg] Convertido → imagenes\ffmpeg\webp\COCO_test2014_000000000171_ffmpeg_q30.webp

🖼 [5] COCO_test2014_000000000173.jpg → webp [imagemagick] calidad=30
[Imag

In [14]:
# Celda 2 — arranca con RAM limpia
df = pd.read_csv("csv/dataset_jpg.csv")
df = procesar_dataset_completo(df, formato_origen="webp")
df.to_csv("csv/dataset_jpg_webp.csv", index=False)
del df
print("✅ WEBP listo")


📁 imagenes\webp  (8000 imágenes total)
   Procesando [0:8000] → 8000 imágenes

🖼 [0] 0 (2).webp → jpg [pillow] calidad=10
Convertido: imagenes\webp\0 (2).webp → imagenes\pillow\jpg\0 (2)_pillow_q10.jpg

🖼 [1] 0044e5aa-e000-4090-8030-00ff5135e73c.webp → jpg [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\0044e5aa-e000-4090-8030-00ff5135e73c_ffmpeg_q10.jpg

🖼 [2] 00a99d2c-01d9-454e-9f7d-26bc066282fc.webp → jpg [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00a99d2c-01d9-454e-9f7d-26bc066282fc_imagemagick_q50.jpg

🖼 [3] 011c6e2a-4618-4b59-a422-0be17fd94294.webp → jpg [pillow] calidad=30
Convertido: imagenes\webp\011c6e2a-4618-4b59-a422-0be17fd94294.webp → imagenes\pillow\jpg\011c6e2a-4618-4b59-a422-0be17fd94294_pillow_q30.jpg

🖼 [4] 017337a4-5a4e-4316-8606-d82acfe81c80.webp → jpg [ffmpeg] calidad=30
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\017337a4-5a4e-4316-8606-d82acfe81c80_ffmpeg_q30.jpg

🖼 [5] 01be242e-6509-4184-968c-752872402d07.webp → jp

In [15]:
# Bloque 1 — primeras 500
df = pd.read_csv("csv/dataset_jpg_webp.csv")
df = procesar_dataset_completo(df, formato_origen="png")
df.to_csv("csv/dataset_completo.csv", index=False)
del df


📁 imagenes\png  (8000 imágenes total)
   Procesando [0:8000] → 8000 imágenes

🖼 [0] 00000000_224resized.png → jpg [pillow] calidad=10
Convertido: imagenes\png\00000000_224resized.png → imagenes\pillow\jpg\00000000_224resized_pillow_q10.jpg

🖼 [1] 00000000_512resized.png → jpg [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000000_512resized_ffmpeg_q10.jpg

🖼 [2] 00000001_224resized.png → jpg [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000001_224resized_imagemagick_q50.jpg

🖼 [3] 00000001_300resized (2).png → jpg [pillow] calidad=30
Convertido: imagenes\png\00000001_300resized (2).png → imagenes\pillow\jpg\00000001_300resized (2)_pillow_q30.jpg

🖼 [4] 00000001_512resized.png → jpg [ffmpeg] calidad=30
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000001_512resized_ffmpeg_q30.jpg

🖼 [5] 00000002_300resized.png → jpg [imagemagick] calidad=30
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000002_300resized_imagemagick_q30.jpg

🖼 [6] 0

c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\wand\image.py:9647: CoderWarning: iCCP: profile 'ICC Profile': 'desc': ICC profile tag start not a multiple of 4 `imagenes\png\00000157_224resized.png' @ warning/png.c/MagickPNGWarningHandler/1531
  self.raise_exception()



🖼 [366] 00000157_512resized (2).png → jpg [pillow] calidad=10
Convertido: imagenes\png\00000157_512resized (2).png → imagenes\pillow\jpg\00000157_512resized (2)_pillow_q10.jpg

🖼 [367] 00000157_512resized.png → jpg [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000157_512resized_ffmpeg_q10.jpg

🖼 [368] 00000158_300resized (2).png → jpg [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000158_300resized (2)_imagemagick_q50.jpg

🖼 [369] 00000158_300resized.png → jpg [pillow] calidad=30
Convertido: imagenes\png\00000158_300resized.png → imagenes\pillow\jpg\00000158_300resized_pillow_q30.jpg

🖼 [370] 00000158_512resized (2).png → jpg [ffmpeg] calidad=50
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000158_512resized (2)_ffmpeg_q50.jpg

🖼 [371] 00000158_512resized.png → jpg [imagemagick] calidad=30
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000158_512resized_imagemagick_q30.jpg

🖼 [372] 00000159_300resized (2).png → jpg [pillow] calida